# Design Spaces 

## Setup

### Import Packages and Directories

In [1]:
from pathlib import Path
import flowtangent as ft
from flowtangent.models.kernels import RBFKernel
from flowtangent.models.surrogates import GaussianProcessSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

In [3]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name:str):

    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(row:dict):
    af_name = row["foil_name"].item()
    row["y_af"] = surf_lookup(af_name)
    return row

### Import Dataset

In [73]:
import os
import zipfile
import jax.numpy as jnp
import numpy as np

# data_source = hf_hub_download(
#     repo_id="jtsmart/XFOIL-4D",
#     repo_type="dataset",
#     filename="xfoil_4D.zarr.zip",
# )

# DATA_DIR = ###Insert Your Local Path Here###

# local_zarr_path = os.path.join(DATA_DIR, "xfoil_4D.zarr")

# if not os.path.exists(local_zarr_path):
#     os.makedirs(DATA_DIR, exist_ok=True)
#     with zipfile.ZipFile(data_source, 'r') as z:
#         z.extractall(DATA_DIR)

XFD = ft.utils.data.XArrayDataset(Path("/Users/jordan/dev/data/xfoil_4D.zarr")).to_tabular(["run", "alpha"])
# XFD = XFD.transform_variable(name2surf)

### Setup Manifold

In [17]:
airfold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
confold = Passthrough(dim=5) # Mach, Re, Flap Angle, Flap Loc., Alpha
mfold = airfold + confold

### Setup Surrogate

In [ ]:
krnl = RBFKernel(input_dim=21)
surr = GaussianProcessSurrogate(42, input_dim=21, output_dim=1, kernel=krnl)

### Setup Parameters and Requirements

In [ ]:
y_af = Parameter(name="y_af")
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Paramter(name="alpha")

theta = (y_af, Mach, re, flap_loc, flap_angle, alpha)

CL_max = Requirement(name="CL", lower_bound=1.2)
alpha_range = Requirement(name="alpha", lower_bound=0.0, upper_bound=5.0)
R = (CL_max, alpha_range)

### Setup Design Space

In [ ]:
airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
    requirements=R
)